In [ ]:
import polars as pl
from importlib import reload
import nbformat

try:
    import plotly.io as pio
    pio.renderers.default = 'notebook_connected'
except ModuleNotFoundError:
    pio = None
    print('plotly is not installed; plots will require `pip install plotly`.')

import config_tbank_dataset as config
reload(config)
from config_tbank_dataset import *

import data_loader_tbank_dataset
reload(data_loader_tbank_dataset)
from data_loader_tbank_dataset import *

from polars_bridge import ensure_pandas_frame

import strategy
reload(strategy)
from strategy import *

import execution_backtester
reload(execution_backtester)

import plotting
reload(plotting)
from plotting import *

import portfolio
reload(portfolio)
from portfolio import *

import rolling_metrics
reload(rolling_metrics)
from rolling_metrics import *

from regime_filters import CombinedRegimeFilter, DrawdownFilter, VolatilityFilter, SpxVixFilter


In [ ]:
print_config_settings()

## Выгрузка данных и формирование весов

Этот notebook повторяет логику `Main_SPX_momentum_weekly.ipynb`, но работает на локальном MOEX dataset из `tbank-trader/data_exports/ru_shares_2022_daily`.

Что меняется по сравнению с SPX-версией:
- `price_table_wo_divs` = **raw daily open** для исполнения
- `price_table_with_divs` = **dividend-adjusted daily close** для ранжирования
- `lot_sizes` берутся по тикерам из локального датасета
- сплиты поддерживаются через optional `splits.csv`, если он появится рядом с dataset
- режимный фильтр по умолчанию локальный drawdown-based, без `SPX/VIX`


In [ ]:
raw_tickers = load_tickers()
lot_sizes_full = load_lot_sizes(raw_tickers)

price_table_wo_divs_raw, price_table_with_divs_raw, divs_raw = fetch_price_data_divs(
    raw_tickers,
    start=START_DATE,
    end=END_DATE,
)
raw_close_prices_raw = get_raw_close_prices(
    raw_tickers,
    start=START_DATE,
    end=END_DATE,
)
turnover_table_raw = get_turnover_table(
    raw_tickers,
    start=START_DATE,
    end=END_DATE,
)
index_price_table_raw = get_benchmark_prices(
    BENCHMARK_TICKERS_LIST,
    start=START_DATE,
    end=END_DATE,
)
status_snapshot = load_status_snapshot()

valid_trading_dates, calendar_report = build_market_calendar(
    price_table_wo_divs_raw,
    min_active_coverage_ratio=CALENDAR_MIN_ACTIVE_COVERAGE_RATIO,
    weekdays_only=CALENDAR_WEEKDAYS_ONLY,
)

price_table_wo_divs = filter_frame_to_dates(price_table_wo_divs_raw, valid_trading_dates)
price_table_with_divs = filter_frame_to_dates(price_table_with_divs_raw, valid_trading_dates)
divs = filter_frame_to_dates(divs_raw, valid_trading_dates)
raw_close_prices = filter_frame_to_dates(raw_close_prices_raw, valid_trading_dates)
turnover_table = filter_frame_to_dates(turnover_table_raw, valid_trading_dates)
index_price_table = filter_frame_to_dates(index_price_table_raw, valid_trading_dates)

eligible_tickers, eligibility_report = build_universe_eligibility(
    price_table=price_table_wo_divs,
    turnover_table=turnover_table,
    min_history_days=UNIVERSE_MIN_HISTORY_DAYS,
    liquidity_window=LIQUIDITY_WINDOW_DAYS,
    min_median_turnover_rub=MIN_MEDIAN_TURNOVER_RUB,
    max_daily_return=MAX_DAILY_RETURN,
    min_daily_return=MIN_DAILY_RETURN,
)

tickers = eligible_tickers
lot_sizes = {ticker: lot_sizes_full[ticker] for ticker in tickers if ticker in lot_sizes_full}

price_table_wo_divs = select_tickers(price_table_wo_divs, tickers)
price_table_with_divs = select_tickers(price_table_with_divs, tickers)
divs = select_tickers(divs, tickers)
raw_close_prices = select_tickers(raw_close_prices, tickers)
turnover_table = select_tickers(turnover_table, tickers)

dataset_summary = {
    'raw_ticker_count': len(raw_tickers),
    'eligible_ticker_count': len(tickers),
    'raw_date_count': price_table_wo_divs_raw.height,
    'valid_date_count': price_table_wo_divs.height,
    'dropped_date_count': price_table_wo_divs_raw.height - price_table_wo_divs.height,
    'corp_action_exclusions': eligibility_report.filter(~pl.col('passes_corp_action_screen')).height,
    'liquidity_exclusions': eligibility_report.filter(~pl.col('passes_liquidity')).height,
    'history_exclusions': eligibility_report.filter(~pl.col('passes_history')).height,
    'benchmark_shape': index_price_table.shape,
    'date_min': str(price_table_wo_divs.select(pl.col('date').min()).item()),
    'date_max': str(price_table_wo_divs.select(pl.col('date').max()).item()),
}
display(pl.DataFrame([dataset_summary]))
display(calendar_report.filter(~pl.col('keep')).head(10))
display(eligibility_report.filter(~pl.col('eligible')).head(20))


In [ ]:
# Рассчет доходностей по периодам (multi-period momentum)
momentum_returns = momentum_calculate_returns(
    price_table_with_divs,
    periods=MOMENTUM_PERIODS,
)
print(f"Momentum periods: {MOMENTUM_PERIODS}")

# Нахождение дат ребалансировок
momentum_rebalance_dates = momentum_get_rebalance_dates(
    momentum_returns,
    freq=REBALANCE_FREQ,
    momentum_period=MOMENTUM_PERIODS[0],
)

# HQM rank scores and target weights
rank_score_dict = momentum_rank_score_calc(momentum_rebalance_dates, momentum_returns)
hqm_table = momentum_hqm_table_calc(rank_score_dict, momentum_returns, rank=MOMENTUM_RANK)
momentum_weights = momentum_weights_table_calc(hqm_table, price_table_with_divs, freq=REBALANCE_FREQ)

print(f"Rebalance dates: {len(momentum_rebalance_dates)}")
print(f"First rebalance date: {momentum_rebalance_dates[0]}")
print(f"Last rebalance date: {momentum_rebalance_dates[-1]}")


In [ ]:
# Веса для дат ребалансировок
target_weights_rebal = momentum_weights.filter(pl.col('date').is_in(momentum_rebalance_dates))
target_weights_rebal.head(5)


## Backtest с учетом фильтров

По умолчанию здесь отключен `SPX/VIX`, потому что notebook должен работать только на локальном MOEX dataset.
Зато можно включать локальные drawdown / volatility filters, не меняя основной execution-based pipeline.


In [ ]:
combined_filter = CombinedRegimeFilter(logic=REGIME_FILTER_LOGIC)

if REGIME_SPX_VIX_ENABLED:
    combined_filter.add_filter(SpxVixFilter(
        enabled=True,
        spx_ma_period=SPX_MA_PERIOD,
        vix_threshold=VIX_THRESHOLD,
        spx_ticker=SPX_TICKER,
        vix_ticker=VIX_TICKER,
    ))

if REGIME_DRAWDOWN_ENABLED:
    combined_filter.add_filter(DrawdownFilter(
        enabled=True,
        window=DRAWDOWN_WINDOW,
        threshold=DRAWDOWN_THRESHOLD,
        source=DRAWDOWN_SOURCE,
    ))

if REGIME_VOLATILITY_ENABLED:
    combined_filter.add_filter(VolatilityFilter(
        enabled=True,
        window=VOLATILITY_WINDOW,
        threshold=VOLATILITY_THRESHOLD,
        source=VOLATILITY_SOURCE,
    ))

combined_filter.get_enabled_filter_names()


In [ ]:
prices_df_exec = ensure_pandas_frame(price_table_wo_divs)
target_weights_exec = ensure_pandas_frame(target_weights_rebal)
benchmark_prices_for_filters = ensure_pandas_frame(index_price_table)

(equity_filtered, returns_filtered, snapshots_filtered, trades_filtered) = execution_backtester.run_execution_backtest_with_filters(
    prices_df=prices_df_exec,
    benchmark_prices_df=benchmark_prices_for_filters,
    target_weights_df=target_weights_exec,
    rebalance_dates=momentum_rebalance_dates,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=lot_sizes,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    apply_cash_yield=(CASH_RETURN_RATE > 0),
    start_date=BACKTEST_START,
    management_fee_monthly=MANAGEMENT_FEE_MONTHLY,
    regime_filter=combined_filter,
)


In [ ]:
display(equity_filtered.head(5).to_frame('equity'))
display(returns_filtered.head(5).to_frame('returns'))
display(snapshots_filtered.head(5))
display(trades_filtered.head(20))

In [ ]:
fig = plot_strategy_equity(
    strategy_equity=equity_filtered,
    rebalance_dates=momentum_rebalance_dates,
    title='MOEX Momentum Equity Curve (Filtered)',
    snapshots_df=snapshots_filtered,
)
fig.show()

expanding_df = calculate_expanding_metrics(equity_filtered)
latest_date = expanding_df.index[-1]

metrics_table = pl.DataFrame({
    'latest_date': [latest_date],
    'equity': [equity_filtered.iloc[-1]],
    'cagr': [expanding_df['cagr'].iloc[-1] if not expanding_df.empty else None],
    'vol': [expanding_df['vol'].iloc[-1] if not expanding_df.empty else None],
    'sharpe': [expanding_df['sharpe'].iloc[-1] if not expanding_df.empty else None],
    'mdd': [expanding_df['mdd'].iloc[-1] if not expanding_df.empty else None],
})
metrics_table


## Сравнение с локальными бенчмарками

Вместо Yahoo-бенчмарков здесь используются локальные бумаги из того же датасета.
По умолчанию это `SBER / LKOH / GAZP`, но список можно менять в `config_tbank_dataset.py`.


In [ ]:
benchmark_lot_sizes = load_lot_sizes(BENCHMARK_TICKERS_LIST)
benchmark_prices_exec = ensure_pandas_frame(index_price_table)
benchmark_results = execution_backtester.run_benchmark_backtest(
    benchmark_tickers=BENCHMARK_TICKERS_LIST,
    prices_df=benchmark_prices_exec,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=benchmark_lot_sizes,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    apply_cash_yield=(CASH_RETURN_RATE > 0),
    start_date=BACKTEST_START,
)

fig = plot_strategy_vs_benchmarks(
    strategy_equity=equity_filtered,
    benchmark_results=benchmark_results,
    rebalance_dates=momentum_rebalance_dates,
    title='MOEX Momentum vs Local Benchmarks',
)
fig.show()


## Расчет позиций для клиентского портфеля

Секция использует те же target weights, но считает реальные позиции с учетом российских lot sizes.


In [ ]:
CLIENT_PORTFOLIO_SIZE = 10000
PORTFOLIO_DATE = None
SAVE_TO_EXCEL = False

positions_df, summary, target_date, output_path = build_client_portfolio_positions(
    weights_df=ensure_pandas_frame(target_weights_rebal),
    prices_df=ensure_pandas_frame(raw_close_prices),
    portfolio_size=CLIENT_PORTFOLIO_SIZE,
    portfolio_date=PORTFOLIO_DATE,
    lot_size=lot_sizes,
    rounding_rule='floor',
    save_to_excel=SAVE_TO_EXCEL,
    output_dir='data',
    verbose=True,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
)
positions_df.head(20)


## Сравнение стратегии с фильтрами / без фильтров


In [ ]:
INITIAL_INVESTMENT

In [ ]:
equity_curve_exec_no_filters, daily_returns_exec_no_filters, snapshots_df_no_filters, trades_df_no_filters = execution_backtester.run_execution_backtest(
    prices_df=prices_df_exec,
    target_weights_df=target_weights_exec,
    rebalance_dates=momentum_rebalance_dates,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=lot_sizes,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    apply_cash_yield=(CASH_RETURN_RATE > 0),
    start_date=BACKTEST_START,
    management_fee_monthly=MANAGEMENT_FEE_MONTHLY,
)

fig = plot_filtered_backtest(
    equity_unfiltered=equity_curve_exec_no_filters,
    equity_filtered=equity_filtered,
    snapshots_filtered=snapshots_filtered,
    rebalance_dates=momentum_rebalance_dates,
    drawdown_threshold=DRAWDOWN_THRESHOLD,
    initial_investment=INITIAL_INVESTMENT,
    title='MOEX Momentum: With vs Without Filters',
)
fig.show()
